# Unsupervised Wellbeing Analysis

This notebook mirrors `unsupervised_wellbeing_analysis.py` step by step so you can see what happens in each cell.

Important scientific note: models are not trained with `normal` / `anomalous` labels. The labels are added only after clustering or anomaly detection to evaluate whether the unsupervised structure corresponds to the known states.


In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

if not os.environ.get('LOKY_MAX_CPU_COUNT'):
    os.environ['LOKY_MAX_CPU_COUNT'] = str(os.cpu_count() or 1)
warnings.filterwarnings('ignore', category=UserWarning)

from unsupervised_wellbeing_analysis import (
    LABELS,
    build_dataset,
    build_loso_anomaly_summary,
    build_summary,
    coerce_feature_columns,
    evaluate_clustering,
    fit_clustering_models,
    format_table,
    infer_group_column,
    inspect_group_candidates,
    load_inputs,
    metadata_columns,
    preprocess_full,
    rows_per_group_table,
    run_anomaly_detection,
    run_anomaly_detection_loso,
    run_feature_ablation,
    save_visualizations,
    summarize_dataset,
    validate_features,
)

DATA_DIR = Path('data/unsupervised')
RESULTS_DIR = Path('results_unsupervised')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOSO_RESULTS_DIR = Path('results/unsupervised_binary')
LOSO_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MPL_CONFIG_DIR = RESULTS_DIR / '.matplotlib'
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG_DIR.resolve()))

RANDOM_STATE = 42
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 100)

print(f'Data directory: {DATA_DIR.resolve()}')
print(f'Results directory: {RESULTS_DIR.resolve()}')
print(f'LOSO anomaly results directory: {LOSO_RESULTS_DIR.resolve()}')


Data directory: /Users/mosi_dev/Developer/university/nearable-project/data/unsupervised
Results directory: /Users/mosi_dev/Developer/university/nearable-project/results_unsupervised
LOSO anomaly results directory: /Users/mosi_dev/Developer/university/nearable-project/results/unsupervised_binary


## 1. Load normal.csv and anomalous.csv

The anomalous file may already contain a `label` column, but the notebook does not use it for training. A separate `true_label` column is created only for evaluation later.


In [2]:
raw_datasets = load_inputs(DATA_DIR)

for name, df in raw_datasets.items():
    print()
    print(f'{name}.csv: {df.shape[0]} rows x {df.shape[1]} columns')
    print(list(df.columns))
    if 'label' in df.columns:
        print('Existing label column counts, not used for training:')
        print(df['label'].value_counts(dropna=False).to_string())



normal.csv: 218 rows x 20 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD']

anomalous.csv: 431 rows x 21 columns
['window_start', 'video_source', 'p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD', 'label']
Existing label column counts, not used for training:
label
anomalous    431


## 2. Inspect participant / source columns

This checks possible source columns such as `video_source` and identifies the group column used only for reporting participant-level counts.


In [3]:
group_inspection = inspect_group_candidates(raw_datasets)
group_col = infer_group_column(raw_datasets)

print(f'Inferred group column for reporting: {group_col}')
for file_name, columns in group_inspection.items():
    print()
    print(f'{file_name}.csv candidate group columns:')
    if not columns:
        print('  none detected')
    for column, values in columns.items():
        print(f'  {column}: {values}')


Inferred group column for reporting: video_source

normal.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']

anomalous.csv candidate group columns:
  video_source: ['Alessia', 'Alvaro', 'Beatrice', 'Mo']


## 3. Validate features and missing values

Only the numerical sensor features are used. Identifiers such as `video_source`, `window_start`, filenames, participant names, original labels, and `true_label` are excluded from model features.


In [4]:
features = validate_features(raw_datasets)
datasets = coerce_feature_columns(raw_datasets, features)

print('Selected numerical feature columns:')
print(features)
print()
print('Note: RMMSD is the CSV column name; scientifically it refers to RMSSD.')
print()
print('Missing values in selected features:')
for name, df in datasets.items():
    missing = df.loc[:, features].isna().sum()
    print()
    print(f'{name}.csv')
    print(missing.to_string())


Selected numerical feature columns:
['p_shoulder_mean', 'p_shoulder_std', 'p_shoulder_bad_pct', 'p_head_tilt_mean', 'p_head_tilt_std', 'p_head_tilt_bad_pct', 'p_fhp_mean', 'p_fhp_std', 'p_fhp_bad_pct', 'b_ear_mean', 'b_ear_std', 'b_perclos', 'b_blink_count', 'b_blink_freq', 'b_ibi_mean', 'b_ibi_std', 'HR', 'RMMSD']

Note: RMMSD is the CSV column name; scientifically it refers to RMSSD.

Missing values in selected features:

normal.csv
p_shoulder_mean        0
p_shoulder_std         0
p_shoulder_bad_pct     0
p_head_tilt_mean       0
p_head_tilt_std        0
p_head_tilt_bad_pct    0
p_fhp_mean             0
p_fhp_std              0
p_fhp_bad_pct          0
b_ear_mean             0
b_ear_std              0
b_perclos              0
b_blink_count          0
b_blink_freq           0
b_ibi_mean             0
b_ibi_std              0
HR                     0
RMMSD                  0

anomalous.csv
p_shoulder_mean        0
p_shoulder_std         0
p_shoulder_bad_pct     0
p_head_tilt_mean     

## 4. Create the combined evaluation dataframe

`true_label` is added here only for evaluation. It is not part of the model feature matrix.


In [5]:
full_df = build_dataset(datasets, group_col)
normal_df = full_df[full_df['true_label'] == 'normal'].reset_index(drop=True)

print(f'Combined rows: {len(full_df)}')
print('True label distribution:')
print(full_df['true_label'].value_counts().to_string())
print()
print('Rows per participant/group per true label:')
if group_col is not None:
    print(format_table(rows_per_group_table(full_df, group_col)))
else:
    print('No group column detected.')


Combined rows: 649
True label distribution:
true_label
anomalous    431
normal       218

Rows per participant/group per true label:
video_source  anomalous  normal  All
     Alessia        110      54  164
      Alvaro        111      54  165
    Beatrice        108      55  163
          Mo        102      55  157
         All        431     218  649


## 5. Preprocess features without labels

The preprocessing pipeline uses median imputation and standard scaling. It receives only the sensor feature columns.


In [6]:
X_scaled, preprocess_pipeline = preprocess_full(full_df, features)
y_true = full_df['true_label'].tolist()

print(f'Feature matrix shape: {full_df.loc[:, features].shape}')
print(f'Scaled matrix shape: {X_scaled.shape}')
print('Preprocessing steps:')
print(preprocess_pipeline)


Feature matrix shape: (649, 18)
Scaled matrix shape: (649, 18)
Preprocessing steps:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


## 6. Clustering experiments

K-Means, Gaussian Mixture, and Agglomerative clustering group the scaled feature matrix without using labels. Afterward, cluster IDs are mapped to labels by majority vote only for evaluation.


In [7]:
cluster_assignments = fit_clustering_models(X_scaled, RANDOM_STATE)
clustering_rows = []
cluster_tables = {}
cluster_predictions = {}
table_filenames = {
    'K-Means': 'kmeans_cluster_label_table.csv',
    'Gaussian Mixture': 'gmm_cluster_label_table.csv',
    'Agglomerative': 'agglomerative_cluster_label_table.csv',
}

for method_name, cluster_ids in cluster_assignments.items():
    row, table, mapped_predictions = evaluate_clustering(
        method_name,
        np.asarray(cluster_ids),
        X_scaled,
        y_true,
        RESULTS_DIR,
        table_filenames[method_name],
    )
    clustering_rows.append(row)
    cluster_tables[method_name] = table
    cluster_predictions[method_name] = mapped_predictions

clustering_results = pd.DataFrame(clustering_rows)
clustering_results.to_csv(RESULTS_DIR / 'clustering_results.csv', index=False)

print('Clustering metrics:')
print(format_table(clustering_results.loc[:, [
    'method',
    'silhouette',
    'adjusted_rand_index',
    'normalized_mutual_info',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
]]))

for method_name, table in cluster_tables.items():
    print()
    print(f'{method_name} cluster vs true-label table:')
    print(table.to_string())


Clustering metrics:
          method  silhouette  adjusted_rand_index  normalized_mutual_info  accuracy  balanced_accuracy  macro_f1
         K-Means    0.344959             0.318015                0.403815  0.782743           0.836427  0.780035
Gaussian Mixture    0.325558             0.392312                0.447832  0.813559           0.859629  0.809736
   Agglomerative    0.340090             0.300519                0.393601  0.775039           0.830626  0.772615

K-Means cluster vs true-label table:
true_label  anomalous  normal mapped_label
cluster                                   
0                 290       0    anomalous
1                 141     218       normal

Gaussian Mixture cluster vs true-label table:
true_label  anomalous  normal mapped_label
cluster                                   
0                 310       0    anomalous
1                 121     218       normal

Agglomerative cluster vs true-label table:
true_label  anomalous  normal mapped_label
cluster     

  File "/Users/mosi_dev/Developer/university/nearable-project/.venv/lib/python3.11/site-packages/joblib/externals/loky/backend/context.py", line 249, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mosi_dev/Developer/university/nearable-project/.venv/lib/python3.11/site-packages/joblib/externals/loky/backend/context.py", line 318, in _count_physical_cores_darwin
    return int(cpu_info)
           ^^^^^^^^^^^^^


## 7. Dimensionality-reduction plots

PCA and t-SNE plots are saved. UMAP is attempted only if it is already installed.


In [8]:
visualization_status = save_visualizations(
    X_scaled,
    y_true,
    cluster_assignments['K-Means'],
    cluster_assignments['Gaussian Mixture'],
    RESULTS_DIR,
    RANDOM_STATE,
)

print('Visualization status:')
print(visualization_status)
print('Saved PCA/t-SNE plots in results_unsupervised/.')


Visualization status:
{'pca_saved': True, 'tsne_saved': True, 'umap_saved': False}
Saved PCA/t-SNE plots in results_unsupervised/.


### PCA and t-SNE plot files

![PCA true labels](results_unsupervised/pca_true_labels.png)

![PCA K-Means clusters](results_unsupervised/pca_kmeans_clusters.png)

![PCA GMM clusters](results_unsupervised/pca_gmm_clusters.png)

![t-SNE true labels](results_unsupervised/tsne_true_labels.png)

![t-SNE K-Means clusters](results_unsupervised/tsne_kmeans_clusters.png)


## 8. Anomaly detection trained only on normal samples

Isolation Forest, One-Class SVM, and Local Outlier Factor are fit only on `normal.csv` samples, then evaluated on `normal.csv + anomalous.csv`.


In [9]:
anomaly_results, anomaly_predictions = run_anomaly_detection(
    normal_df,
    full_df,
    features,
    RESULTS_DIR,
    RANDOM_STATE,
)
anomaly_results.to_csv(RESULTS_DIR / 'anomaly_detection_results.csv', index=False)

print('Anomaly detection metrics:')
print(format_table(anomaly_results.loc[:, [
    'method',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
    'weighted_f1',
    'precision_macro',
    'recall_macro',
]]))

for method_name, predictions in anomaly_predictions.items():
    print()
    print(f'{method_name} classification report:')
    print(classification_report(y_true, predictions, labels=LABELS, zero_division=0))


Anomaly detection metrics:
            method  accuracy  balanced_accuracy  macro_f1  weighted_f1  precision_macro  recall_macro
   IsolationForest  0.983051           0.974771  0.980759     0.982938         0.987557      0.974771
       OneClassSVM  0.973806           0.961009  0.970044     0.973528         0.981027      0.961009
LocalOutlierFactor  0.986133           0.979358  0.984295     0.986058         0.989773      0.979358

IsolationForest classification report:
              precision    recall  f1-score   support

      normal       1.00      0.95      0.97       218
   anomalous       0.98      1.00      0.99       431

    accuracy                           0.98       649
   macro avg       0.99      0.97      0.98       649
weighted avg       0.98      0.98      0.98       649


OneClassSVM classification report:
              precision    recall  f1-score   support

      normal       1.00      0.92      0.96       218
   anomalous       0.96      1.00      0.98       431

## 9. Leave-One-Subject-Out anomaly detection

This is the scientifically stronger anomaly-detection evaluation. For each held-out participant, preprocessing and the anomaly detector are fitted only on normal samples from the other participants. The held-out participant's normal + anomalous samples are then evaluated. Labels are used only afterward for metrics.


In [10]:
loso_results, loso_predictions, loso_prediction_dict = run_anomaly_detection_loso(
    full_df,
    features,
    group_col,
    LOSO_RESULTS_DIR,
    RANDOM_STATE,
)
loso_results.to_csv(LOSO_RESULTS_DIR / 'anomaly_detection_loso_results.csv', index=False)
loso_predictions.to_csv(LOSO_RESULTS_DIR / 'anomaly_detection_loso_predictions.csv', index=False)

loso_summary = build_loso_anomaly_summary(
    loso_results,
    loso_predictions,
    anomaly_results,
)
(LOSO_RESULTS_DIR / 'anomaly_detection_loso_summary.txt').write_text(
    loso_summary + chr(10),
    encoding='utf-8',
)

print('LOSO anomaly detection fold and overall metrics:')
print(format_table(loso_results.loc[:, [
    'row_type',
    'method',
    'fold',
    'held_out_group',
    'train_normal_samples',
    'test_normal_samples',
    'test_anomalous_samples',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
    'weighted_f1',
    'precision_normal',
    'recall_normal',
    'precision_anomalous',
    'recall_anomalous',
]]))
print()
print(loso_summary)
print(f'Saved LOSO anomaly outputs to: {LOSO_RESULTS_DIR.resolve()}')


LOSO anomaly detection fold and overall metrics:
row_type             method  fold held_out_group  train_normal_samples  test_normal_samples  test_anomalous_samples  accuracy  balanced_accuracy  macro_f1  weighted_f1  precision_normal  recall_normal  precision_anomalous  recall_anomalous
    fold    IsolationForest     1        Alessia                 164.0                   54                     110  0.804878           0.703704  0.725982     0.776188               1.0       0.407407             0.774648               1.0
    fold    IsolationForest     2         Alvaro                 164.0                   54                     111  0.787879           0.675926  0.692181     0.751472               1.0       0.351852             0.760274               1.0
    fold    IsolationForest     3       Beatrice                 163.0                   55                     108  0.907975           0.863636  0.888585     0.903698               1.0       0.727273             0.878049          

## 10. Feature ablation

K-Means and Isolation Forest are repeated with posture-only, eye/blink-only, HR/HRV-only, and all-feature groups.


In [11]:
ablation_results = run_feature_ablation(normal_df, full_df, RANDOM_STATE)
ablation_results.to_csv(RESULTS_DIR / 'unsupervised_feature_ablation.csv', index=False)

print('Feature ablation results:')
print(format_table(ablation_results.loc[:, [
    'feature_group',
    'method',
    'n_features',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
    'precision_macro',
    'recall_macro',
    'silhouette',
]]))


Feature ablation results:
 feature_group          method  n_features  accuracy  balanced_accuracy  macro_f1  precision_macro  recall_macro  silhouette
  posture_only         K-Means           9  0.801233           0.850348  0.797846         0.814121      0.850348    0.436871
  posture_only IsolationForest           9  0.981510           0.973611  0.979035         0.985124      0.973611         NaN
eye_blink_only         K-Means           7  0.758089           0.814465  0.755861         0.785971      0.814465    0.399122
eye_blink_only IsolationForest           7  0.893683           0.907485  0.886240         0.876243      0.907485         NaN
   hr_hrv_only         K-Means           2  0.664099           0.500000  0.399074         0.332049      0.500000    0.538507
   hr_hrv_only IsolationForest           2  0.919877           0.927207  0.912956         0.903623      0.927207         NaN
  all_features         K-Means          18  0.782743           0.836427  0.780035         0.803621 

## 11. Save per-row predictions

This file includes metadata columns, `true_label`, cluster IDs, mapped cluster labels, and anomaly-detection predictions.


In [12]:
predictions_df = full_df.loc[:, metadata_columns(full_df, features)].copy()
predictions_df['true_label'] = full_df['true_label']
predictions_df['kmeans_cluster'] = cluster_assignments['K-Means']
predictions_df['kmeans_mapped_label'] = cluster_predictions['K-Means']
predictions_df['gmm_cluster'] = cluster_assignments['Gaussian Mixture']
predictions_df['gmm_mapped_label'] = cluster_predictions['Gaussian Mixture']
predictions_df['agglomerative_cluster'] = cluster_assignments['Agglomerative']
predictions_df['agglomerative_mapped_label'] = cluster_predictions['Agglomerative']
predictions_df['isolation_forest_prediction'] = anomaly_predictions['IsolationForest']
predictions_df['oneclass_svm_prediction'] = anomaly_predictions['OneClassSVM']
predictions_df['lof_prediction'] = anomaly_predictions['LocalOutlierFactor']
predictions_df.to_csv(RESULTS_DIR / 'unsupervised_predictions.csv', index=False)

print(f'Prediction dataframe shape: {predictions_df.shape}')
print(list(predictions_df.columns))
print()
print(predictions_df.head().to_string(index=False))


Prediction dataframe shape: (649, 15)
['source_file', 'source_row', 'window_start', 'video_source', 'original_label', 'true_label', 'kmeans_cluster', 'kmeans_mapped_label', 'gmm_cluster', 'gmm_mapped_label', 'agglomerative_cluster', 'agglomerative_mapped_label', 'isolation_forest_prediction', 'oneclass_svm_prediction', 'lof_prediction']

source_file  source_row  window_start video_source original_label true_label  kmeans_cluster kmeans_mapped_label  gmm_cluster gmm_mapped_label  agglomerative_cluster agglomerative_mapped_label isolation_forest_prediction oneclass_svm_prediction lof_prediction
 normal.csv           0           0.0       Alvaro            NaN     normal               1              normal            1           normal                      1                     normal                   anomalous                  normal         normal
 normal.csv           1          10.0       Alvaro            NaN     normal               1              normal            1           norm

## 12. Final summary and interpretation

This writes `results_unsupervised/final_unsupervised_summary.txt` and prints the final interpretation.


In [13]:
dataset_lines = summarize_dataset(
    datasets,
    full_df,
    features,
    group_col,
    group_inspection,
)
summary_text = build_summary(
    dataset_lines,
    clustering_results,
    anomaly_results,
    ablation_results,
    cluster_tables,
    anomaly_predictions,
    full_df,
    visualization_status,
)
summary_path = RESULTS_DIR / 'final_unsupervised_summary.txt'
summary_path.write_text(summary_text + chr(10), encoding='utf-8')

final_start = summary_text.index('FINAL INTERPRETATION')
print(summary_text[final_start:])
print()
print(f'Saved final summary to: {summary_path.resolve()}')
print(f'Saved unsupervised outputs to: {RESULTS_DIR.resolve()}')

print(f'Saved LOSO anomaly outputs to: {LOSO_RESULTS_DIR.resolve()}')


FINAL INTERPRETATION
Clustering showed strong alignment with normal/anomalous labels.
Best clustering method: Gaussian Mixture (macro F1=0.810, balanced accuracy=0.860, ARI=0.392, NMI=0.448).
Best anomaly detection method: LocalOutlierFactor (macro F1=0.984, balanced accuracy=0.979).
HR/RMSSD dominance check: HR/HRV-only macro F1 was 0.399 for K-Means and 0.913 for Isolation Forest. Compare these with the all-feature and other modality rows in the ablation table.
K-Means: best feature group was posture_only (macro F1=0.798, balanced accuracy=0.850); the best single-modality feature set outperformed all-feature fusion. HR/HRV-only macro F1=0.399.
IsolationForest: best feature group was all_features (macro F1=0.981, balanced accuracy=0.975); all-feature fusion improved over the best single-modality feature set. HR/HRV-only macro F1=0.913.
This is proof-of-concept only because the dataset contains samples from only 4 participants; more participants are needed for stronger generalization c